# Body Side Evaluator — Calibration

Owner: member B — this notebook belongs to one person, so editing it never conflicts with others.

`BodySideEvaluator` (`evaluation/eval_body_side.py`) decides **which side of the body — left / right / both — performs the action**. Left/right come from anatomical joint labels, not the X axis; evidence is limb activity (wrist/ankle deviation from the neutral hanging pose, normalised by limb length) and the laterality index LI = (A_left − A_right)/(A_left + A_right). Full method: docstring of `eval_body_side.py`.

Human Gold Labels are read from `labels/<MODEL_NAME>_pilot_human_gold_labels.json` (create them in STEP 9D/9E of `run_benchmark.ipynb`).

## STEP 0 — Get the code from GitHub

Clones the repository (first run) or pulls the latest version, then imports `evaluation/common.py` and **every `evaluation/eval_*.py` automatically** — a new evaluator file is picked up without editing this notebook.

- `BRANCH = "main"` for normal use; set it to your branch name to test your work before it is merged.
- Private repository only: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon, left sidebar).
- CPU runtime is enough for evaluation (no GPU needed).

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Soniaaaa-aa/t2m-capability-benchmark"
BRANCH = "main"                                   # or your feature branch
REPO_DIR = Path("/content/t2m-capability-benchmark")

def _git(*args, cwd=None):
    print("$ git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True)

url = REPO_URL
try:  # optional token for a private repository
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if not (REPO_DIR / ".git").exists():
    _git("clone", "--branch", BRANCH, url, str(REPO_DIR))
else:
    _git("fetch", "origin", cwd=REPO_DIR)
    _git("checkout", BRANCH, cwd=REPO_DIR)
    _git("pull", "origin", BRANCH, cwd=REPO_DIR)

EVAL_DIR = REPO_DIR / "evaluation"
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

import common
importlib.reload(common)                 # pick up changes after a pull
from common import *                     # settings, loaders, registry, gold helpers, runner
evaluator_modules = load_all_evaluators(EVAL_DIR)   # imports every eval_*.py

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("Framework commit     :", commit or "unknown")

## STEP 1 — Load the model's motions, cases and Human Gold Labels

One call replaces STEP 3.5–9F of `run_benchmark.ipynb` (upload prompt appears only if the ZIP / benchmark JSON is missing).

In [ ]:
MODEL_NAME = "MoMADiff"      # ← model to analyse

data = load_inputs(MODEL_NAME, REPO_DIR)
evaluation_cases = data["evaluation_cases"]
HUMAN_GOLD_LABELS = data["human_gold_labels"]

In [ ]:
import eval_body_side

## STEP 2 — Body Side Evidence vs Human Gold

In [ ]:
body_side_results = eval_body_side.collect_body_side_evidence(evaluation_cases, HUMAN_GOLD_LABELS)

print("=" * 100)
print(f"BODY SIDE EVIDENCE — {MODEL_NAME}")
print("=" * 100)
header = f"{'Prompt':<7} {'Req':<6} {'Limb':<5} {'A_left':>7} {'A_right':>8} {'LI':>7} {'Raw':<12} {'Human':<10} Source"
print(header)
print("-" * len(header))
for r in body_side_results:
    print(f"{r['prompt_id']:<7} {r['required_side']:<6} {r['limb']:<5} "
          f"{r['left_activity']:>7.2f} {r['right_activity']:>8.2f} {r['laterality_index']:>+7.2f} "
          f"{r['predicted_side_raw']:<12} {str(r['human_label']):<10} {r['limb_source']}")
labels = [r["human_label"] for r in body_side_results]
print("\nHuman labels — PASS:", labels.count("PASS"), "| FAIL:", labels.count("FAIL"),
      "| UNCERTAIN:", labels.count("UNCERTAIN"), "| missing:", labels.count(None))

## STEP 3 — Threshold Calibration

Grid search against Human Gold (`UNCERTAIN` excluded): highest accuracy → fewest false PASS → closest to the provisional values. Fewer than 10 labelled cases → `calibrated_small_sample`.

In [ ]:
calibration = eval_body_side.calibrate_body_side_thresholds(body_side_results)
BODY_SIDE_THRESHOLDS = calibration["thresholds"]
BODY_SIDE_THRESHOLD_STATUS = calibration["status"]

print("=" * 100)
print("BODY SIDE THRESHOLD CALIBRATION")
print("=" * 100)
print("Usable Human-labelled cases:", calibration["usable"])
if calibration["usable"]:
    print(f"Best accuracy: {calibration['correct']}/{calibration['usable']} "
          f"| false PASS: {calibration['false_pass']} "
          f"| combinations reaching it: {calibration['n_best']} of {calibration['grid_size']}")
    for r in calibration["cases"]:
        mark = "✅" if r["predicted"] == r["human_label"] else "❌"
        print(f"  {r['prompt_id']:<7} required={r['required_side']:<5} human={r['human_label']:<4} "
              f"predicted={r['predicted']:<4} {mark}")
else:
    print("No PASS / FAIL labels — provisional thresholds kept.")
print("\nSelected thresholds:", BODY_SIDE_THRESHOLDS)
print("Threshold status   :", BODY_SIDE_THRESHOLD_STATUS)
print("Currently used by run_benchmark.ipynb:", eval_body_side.BodySideEvaluator.CURRENT_THRESHOLDS)

## STEP 4 — Evaluation with the Selected Thresholds

In [ ]:
body_side_final_results = eval_body_side.evaluate_body_side(
    evaluation_cases, BODY_SIDE_THRESHOLDS, BODY_SIDE_THRESHOLD_STATUS, HUMAN_GOLD_LABELS)

for r in body_side_final_results:
    agree = ""
    if r["human_label"] in {"PASS", "FAIL"}:
        agree = "✅" if r["pass_fail"] == r["human_label"] else "❌"
    print(f"{r['prompt_id']:<7} body_side={r['expected_value']:<5} -> {r['pass_fail']:<4} "
          f"(predicted {r['predicted_side']}, human {r['human_label']}) {agree}")
    print(f"        {r['reason']}")
labelled = [r for r in body_side_final_results if r["human_label"] in {"PASS", "FAIL"}]
if labelled:
    print(f"\nAgreement with Human Gold: {sum(r['pass_fail'] == r['human_label'] for r in labelled)}/{len(labelled)}")

## STEP 5 — Adopt the thresholds

When the thresholds are confirmed across the Pilot models, copy them into `CURRENT_THRESHOLDS` (and update `CURRENT_THRESHOLD_STATUS`) in `evaluation/eval_body_side.py` through a Pull Request. `run_benchmark.ipynb` then uses them automatically.